# 🇹🇭 Challenge 5: Thai Image Captioning

**โจทย์:** สร้างคำบรรยายภาพเป็นภาษาไทย จากชุดภาพอาหารและท่องเที่ยวของไทย (IPU24)

**วิธีการ:** ใช้โมเดล `thaicapgen-swin-wangchan` (SwinV2 encoder + WangchanBERTa decoder) พร้อม beam search

**สิ่งที่ต้องเตรียม:**
1. Kaggle API Token (`kaggle.json`) — สำหรับดาวน์โหลดข้อมูลแข่งขัน
2. โฟลเดอร์โมเดลใน Google Drive — อัปโหลดโฟลเดอร์ `thaicapgen-swin-wangchan` ไปที่ `MyDrive/models/`

## 1. ติดตั้ง libraries ที่จำเป็น

In [ ]:
!pip install -q transformers sentencepiece protobuf kaggle

## 2. ดาวน์โหลดข้อมูลจาก Kaggle

อัปโหลดไฟล์ `kaggle.json` แล้วดาวน์โหลดชุดข้อมูลการแข่งขัน

In [ ]:
import os
from google.colab import files

# อัปโหลด kaggle.json
uploaded = files.upload()  # เลือกไฟล์ kaggle.json

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

# ดาวน์โหลดข้อมูลการแข่งขัน
COMPETITION = "super-ai-engineer-ss-6-thai-language-image-captioning"
DATA_DIR = f"/content/{COMPETITION}"

!kaggle competitions download -c {COMPETITION} -p /content/
!mkdir -p {DATA_DIR}
!unzip -qo /content/{COMPETITION}.zip -d {DATA_DIR}

print(f"\n✅ ดาวน์โหลดข้อมูลเสร็จ: {DATA_DIR}")
!ls {DATA_DIR}

## 3. เชื่อมต่อ Google Drive และโหลดโมเดล

> **ขั้นตอนก่อนหน้า:** อัปโหลดโฟลเดอร์ `thaicapgen-swin-wangchan` (ที่มี `config.json`, `model.safetensors`, `modeling_cap.py`, ฯลฯ) ไปที่ `Google Drive > MyDrive > models > thaicapgen-swin-wangchan`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MODEL_DIR = "/content/drive/MyDrive/models/thaicapgen-swin-wangchan"

# ตรวจสอบว่าไฟล์โมเดลมีครบ
required = ["config.json", "model.safetensors", "modeling_cap.py",
            "configuration_cap.py", "tokenizer.json", "preprocessor_config.json"]
for f in required:
    assert os.path.exists(os.path.join(MODEL_DIR, f)), f"❌ ไม่พบไฟล์: {f}"

print("✅ พบไฟล์โมเดลครบ")
!ls {MODEL_DIR}

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ ใช้ device: {device}")

# โหลดโมเดล (trust_remote_code เพราะใช้ custom modeling_cap.py)
processor = AutoImageProcessor.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True)
model = model.to(device).eval()

print(f"✅ โหลดโมเดลสำเร็จ — params: {sum(p.numel() for p in model.parameters()):,}")

## 4. กำหนด paths และ parameters

In [ ]:
import re
from pathlib import Path

import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# paths ข้อมูลแข่งขัน
TEST_DIR = Path(DATA_DIR) / "test" / "test"
SAMPLE_SUB = Path(DATA_DIR) / "sample_submission.csv"
OUTPUT_CSV = "/content/submission.csv"

# ตั้งค่า beam search
NUM_BEAMS = 5      # จำนวน beams (ยิ่งมาก = ผลดีขึ้นแต่ช้าลง)
MAX_LENGTH = 80    # ความยาว caption สูงสุด (tokens)

# ตรวจสอบข้อมูล
sample_sub = pd.read_csv(SAMPLE_SUB)
test_images = sorted(TEST_DIR.glob("*.jpg"))
print(f"📊 จำนวนรูปใน template: {len(sample_sub)}")
print(f"📁 จำนวนรูปใน test dir: {len(test_images)}")

## 5. ฟังก์ชันสร้างคำบรรยายภาพ

In [ ]:
def normalize_caption(text: str) -> str:
    """ทำความสะอาดข้อความ caption ภาษาไทย"""
    x = (text or "").strip()
    x = x.replace("\u0e4d\u0e32", "\u0e33")   # แก้สระอำ
    x = x.replace("\u0e40\u0e40", "\u0e41")   # แก้สระแอ
    x = re.sub(r"\s+", " ", x)
    return x.strip(" .,;:")


@torch.inference_mode()
def generate_caption(img_path: Path) -> str:
    """สร้าง caption ภาษาไทยจากภาพ 1 รูป"""
    image = Image.open(img_path).convert("RGB")
    inputs = processor(images=[image], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    output_ids = model.generate(
        **inputs,
        max_length=MAX_LENGTH,
        num_beams=NUM_BEAMS,
        use_cache=False,
    )
    text = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]
    return normalize_caption(text)


# ทดสอบกับ 1 รูป
test_caption = generate_caption(test_images[0])
print(f"🖼️ ทดสอบ {test_images[0].name}: {test_caption}")

## 6. สร้าง caption สำหรับรูปทดสอบทั้งหมด (2,000 รูป)

ใช้เวลาประมาณ 5-10 นาทีบน T4 GPU

In [ ]:
import time

# อ่าน image_id จาก template เพื่อให้ลำดับตรงกัน
ids = [str(x).zfill(5) for x in sample_sub["image_id"].tolist()]

FALLBACK = "ภาพอาหารหรือสถานที่ท่องเที่ยวที่ปรากฏอยู่ในภาพ"
results = {}
failures = 0
t0 = time.time()

for i, img_id in enumerate(ids, 1):
    img_path = TEST_DIR / f"{img_id}.jpg"
    try:
        results[img_id] = generate_caption(img_path)
    except Exception as e:
        results[img_id] = FALLBACK
        failures += 1
        print(f"  ⚠️ {img_id}: {e}")

    # แสดงความคืบหน้าทุก 100 รูป
    if i % 100 == 0 or i == len(ids):
        elapsed = time.time() - t0
        rate = i / elapsed
        eta = (len(ids) - i) / rate
        print(f"  [{i:>4}/{len(ids)}] {elapsed:.0f}s | ~{eta:.0f}s เหลือ | ผิดพลาด: {failures}")

print(f"\n✅ เสร็จสิ้น: {len(results)} captions ใน {time.time()-t0:.1f}s (ผิดพลาด: {failures})")

## 7. สร้างไฟล์ submission.csv

In [ ]:
# สร้าง DataFrame โดยเรียง image_id ตาม template เดิม
rows = []
for row in sample_sub.itertuples(index=False):
    img_id = str(row.image_id).zfill(5)
    caption = results.get(img_id, FALLBACK)
    rows.append({"image_id": row.image_id, "caption": caption})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(OUTPUT_CSV, index=False)
print(f"💾 บันทึกไฟล์: {OUTPUT_CSV}")
sub_df.head(10)

## 8. ตรวจสอบความถูกต้อง

In [ ]:
# ตรวจสอบว่า submission ตรงกับ template ทุกประการ
template_ids = sample_sub["image_id"].tolist()
submit_ids = sub_df["image_id"].tolist()

assert len(sub_df) == len(sample_sub), f"❌ จำนวนแถวไม่ตรง: {len(sub_df)} vs {len(sample_sub)}"
assert template_ids == submit_ids,     "❌ ลำดับ image_id ไม่ตรง"
assert sub_df["caption"].str.strip().ne("").all(), "❌ พบ caption ว่าง"

# สรุปผล
thai_re = re.compile(r"[\u0E00-\u0E7F]")
thai_pct = sub_df["caption"].apply(lambda x: len(thai_re.findall(x)) >= 5).mean() * 100

print(f"✅ จำนวนแถว: {len(sub_df)} (ตรงกับ template)")
print(f"✅ image_id: ลำดับตรงกันทั้งหมด")
print(f"✅ ไม่มี caption ว่าง")
print(f"✅ มีอักษรไทย ≥5 ตัว: {thai_pct:.1f}%")

## 9. ดาวน์โหลดไฟล์ submission

In [ ]:
files.download(OUTPUT_CSV)
print("📥 ดาวน์โหลด submission.csv เสร็จเรียบร้อย!")